In [ ]:
%cd ..

# import libraries

In [ ]:
import pytesseract
from pdf2image import convert_from_path
from tqdm import tqdm 


# OCR PDF

In [ ]:

def ocr_book(pdf_path, output_file):
    pages = convert_from_path(pdf_path, 300, first_page=601, last_page=840) 

    full_text = []

    for i, page in enumerate(tqdm(pages)):
        text = pytesseract.image_to_string(page, lang='eng')
        
        full_text.append(text)

    with open(output_file, "w", encoding="utf-8") as f:
        f.writelines(full_text)

In [ ]:
ocr_book("Vector-rag 2/book/pdf/Bleak_House.pdf", "Vector-rag 2/book/text/Bleak_House_3.txt")

# compil into a single file

In [ ]:
files = ["Vector-rag 1/books/text/Bleak_House_1.txt", 
         "Vector-rag 1/books/text/Bleak_House_2.txt", 
         "Vector-rag 1/books/text/Bleak_House_3.txt"
         ]

with open("Vector-rag 1/books/text/merged.txt", "w", encoding="utf-8") as outfile:
    for fname in files:
        with open(fname, "r", encoding="utf-8") as infile:
            outfile.write(infile.read())
            outfile.write("\n") 

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.runnables import RunnablePassthrough
from sentence_transformers import SentenceTransformer
import torch


# Vector store with Chroma

In [ ]:
file_path = "Vector-rag 1/books/text/merged.txt" 
with open(file_path, "r", encoding="utf-8") as f:
    full_text = f.read()


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", "مادة", " "]
)
all_chunks = text_splitter.split_text(full_text)


In [ ]:


class Embedding: 
    def __init__(self):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model = SentenceTransformer('intfloat/multilingual-e5-small', device=self.device)
    
    def embed_documents(self, docs):
        processed_docs = [f"passage: {doc}" for doc in docs]
        embeddings = self.model.encode(processed_docs, device=self.device, show_progress_bar=True)
        return embeddings.tolist()
    
    def embed_query(self, query):
        processed_query = f"query: {query}"
        embeddings = self.model.encode([processed_query], device=self.device)
        return embeddings.tolist()[0]

In [ ]:
custom_embeddings = Embedding()

In [ ]:
vector_db = Chroma.from_texts(
    texts=all_chunks, 
    embedding=custom_embeddings, 
    persist_directory="./Vector-rag 1/vector_db"
)

In [ ]:
db_check = Chroma(
    persist_directory="Vector-rag 1/vector_db", 
    embedding_function=custom_embeddings
)

In [ ]:
retreiver = db_check.as_retriever(search_type='similarity', search_kwargs={'k': 3})

In [ ]:
"What are the different locations mentioned in the text where the fog is present, and how does it affect the people there?"
"Who are the people mentioned in the documents?"

In [ ]:
print(retreiver.invoke("What are the different locations mentioned in the text where the fog is present, and how does it affect the people there?")[1].page_content)

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:


llm = ChatOllama(
    model="gemma3:4b",
    temperature=0, 
)

In [ ]:
template = """You are an intelligent assistant specialized in analyzing literary texts, especially the novel "Black Beauty" by Anna Sewell.

Your task is to answer questions strictly based on the provided "context". Do not use any external knowledge.

Follow these rules carefully:

1. Only use the given context to answer.
2. If the answer is not found in the context, respond with:
   "There is not enough information in the text to answer this question."

3. When answering:
   - Use clear and simple English.
   - Stay faithful to the tone and meaning of the story.

4. If the question is about a character:
   - Describe their role in the story.
   - Mention their traits or behavior based on the text.

5. If the question is about an event:
   - Summarize what happened briefly.
   - Explain its impact on the story or on Black Beauty.

6. If the question is analytical:
   - Identify themes or messages (e.g., kindness, cruelty, animal welfare, responsibility).

7. If the question involves emotions:
   - Describe the feelings of Black Beauty or other characters.
   - Connect those feelings to the situation in the text.

Structure your answer as follows:

🔹 Answer:
(Direct answer)

🔹 Explanation:
(Supporting details from the context)

---
Question: {question}

Contextual excerpts from the book:
{context}

Detailed Literary Answer:
"""

prompt = PromptTemplate.from_template(template)

In [ ]:

rag_chain = (
    {"context": retreiver | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
"Compare Black Beauty and Ginger in terms of personality, experiences, and reactions to human treatment."
"How does Black Beautys early life influence his behavior later in the story?"
"What message does the novel convey about the relationship between humans and animals? Support your answer with examples."
"Do you think the suffering of Black Beauty was caused more by ignorance or cruelty? Explain your reasoning based on the story."
"How can the lessons from Black Beauty be applied to modern society, especially in how animals are treated today?"

In [ ]:
answer = rag_chain.invoke("How can the lessons from Black Beauty be applied to modern society, especially in how animals are treated today?")
print(answer)

In [ ]:
import gradio as gr

def chat(message, history):
    result = rag_chain.invoke({"question": message})
    return result

demo = gr.ChatInterface(
    fn=chat, 
    title="🐴 Black Beauty",
    examples=[
        "Who was Black Beauty's first kind master?",
        "Describe the 'Ginger' character and her relationship with Beauty.",
        "What are the main themes of animal welfare in the book?",
        "Tell me about the time Beauty lived in London as a cab horse."
    ],
)

demo.launch()

In [ ]:
gr.close_all()